<div style="background:#0b3d91;color:white;padding:40px;text-align:center;border-radius:8px;">

# **German Traffic Sign Recognition**
## *A Friendly Walk-Through of Our Deep-Learning Project*

### Abdulrahman Aqili &nbsp; & &nbsp; Ali Al-Qarni
#### Course: Deep Learning

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **1. What is this project about?**

We taught a computer to **look at a photo of a traffic sign and say what it is** — for example *Stop*, *Yield*, or *Speed limit 70*.

This is the same kind of technology used in **self-driving cars** and **driver-assistance systems** to read road signs in real time.

Our goal is simple to state but not easy to solve:

> Given a small photo of a sign, predict the correct category out of **43 possible traffic signs**.

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **2. The data — GTSRB**

We use the **German Traffic Sign Recognition Benchmark (GTSRB)** — a famous public dataset.

- **43 classes** of real traffic signs.
- About **39,000** training images and **12,000** test images.
- Photos were taken from a real car in real driving conditions: different **lighting, angles, blur, weather**.
- Some classes have many more pictures than others (**class imbalance**) — the model must still learn the rare ones.

We resize every image to **64 × 64 pixels** so the model gets inputs of the same size.

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **3. How we prepared the images**

Before training, every image goes through the same pipeline:

1. **Resize** to 64 × 64.
2. **Normalize** the colors (subtract mean, divide by std) so the network learns faster.

On the **training set only**, we add **data augmentation** — small random changes to each image so the model never sees the exact same picture twice:

- Slight **rotation**, **shift**, **zoom**, and **shear** — signs in real life are rarely perfectly aligned.
- Random **brightness / contrast / saturation** — to mimic sun, shadow and weather.

Augmentation makes the model **more robust** and reduces overfitting.

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **4. The tools**

- **PyTorch** — the deep-learning framework.
- **torchvision** — built-in datasets, image transforms, and pre-trained models.
- **scikit-learn** — evaluation metrics (confusion matrix, classification report).
- **matplotlib** — charts and prediction visualizations.

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **5. Model 1 — Custom CNN (built from scratch)**

Our first model is a small **Convolutional Neural Network** we designed ourselves. It has three convolutional blocks that learn richer visual features step by step:

- **Block 1** → learns basic **edges and colors**.
- **Block 2** → learns **shapes and corners**.
- **Block 3** → learns **sign-like patterns**.

After the blocks we use **Global Average Pooling** + a small classifier that outputs one score per class (43 scores).

To keep the network healthy:

- **BatchNorm** after each convolution → stable training.
- **Dropout** → fights overfitting.
- **ReLU** activations → fast and avoid vanishing gradients.

</div>


In [ ]:
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super().__init__()
        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch,  out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.MaxPool2d(2), nn.Dropout(0.25),
            )
        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.gap(x)
        return self.classifier(x)


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **6. Model 2 — ResNet-50 with Transfer Learning**

Instead of training a giant model from zero, we **reuse a model that already learned from millions of images** (ImageNet).

The idea is called **Transfer Learning**:

1. Load **ResNet-50** with its pre-trained weights.
2. **Freeze** all original layers — keep the visual knowledge they already have.
3. **Replace the final layer** with a new one that outputs **43 classes** (our traffic signs).
4. Train **only** that final layer.

The result: **less training time** and **higher accuracy** than training from scratch.

</div>


In [ ]:
resnet_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
for p in resnet_model.parameters():
    p.requires_grad = False
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, NUM_CLASSES)
resnet_model = resnet_model.to(DEVICE)


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **7. How we trained**

- **Loss:** Cross-Entropy with **label smoothing** (model is not over-confident).
- **Optimizer:** Adam with weight decay.
- **Scheduler:** Cosine annealing — the learning rate decreases smoothly over time.
- **Gradient clipping** at 1.0 — keeps updates stable.
- **Early stopping** (patience = 5) — stops automatically when the model stops improving.
- We always save the **best model** (best validation accuracy) to disk.

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **8. Results — training curves**

Both models converge quickly. **ResNet-50** starts higher and reaches a lower loss because of its pre-trained features.

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **9. Sample predictions on real test images**

For each test image we show the **picture** and a clear caption such as:

> **This sign indicates: "Speed limit (70km/h)" — Confidence: 98.4%**

A **green** title = the model was **correct**. A **red** title = the model was wrong (and we also print the true class).

</div>


In [ ]:
# Sample predictions on the test set
# For each image we print the picture and a caption like:
#   "This sign indicates: Speed limit (70km/h)   (98.4%)"
# Green title = correct prediction, Red title = wrong prediction.

cnn_model.load_state_dict(torch.load('best_cnn.pt',    map_location=DEVICE))
resnet_model.load_state_dict(torch.load('best_resnet.pt', map_location=DEVICE))
cnn_model.eval(); resnet_model.eval()

sample_ids = random.sample(range(len(test_raw)), 8)

def show_predictions(model, model_name, header_color):
    fig, axes = plt.subplots(2, 4, figsize=(16, 9))
    fig.suptitle(f'{model_name} — Sample Test Predictions',
                 fontsize=15, fontweight='bold', color=header_color)
    for ax, idx in zip(axes.flat, sample_ids):
        img_pil, true_label = test_raw[idx]
        x = eval_tf(img_pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(model(x), dim=1)[0].cpu().numpy()
        pred = int(probs.argmax())
        conf = float(probs[pred]) * 100
        correct = (pred == true_label)

        ax.imshow(img_pil)
        ax.axis('off')
        caption = f'This sign indicates:\n"{CLASS_NAMES[pred]}"\nConfidence: {conf:.1f}%'
        if not correct:
            caption += f'\n(True: {CLASS_NAMES[true_label]})'
        ax.set_title(caption, fontsize=10,
                     color=('#1e8449' if correct else '#c0392b'),
                     fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(cnn_model,    'Custom CNN', '#2874a6')
show_predictions(resnet_model, 'ResNet-50',  '#b9770e')


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **10. Final comparison**

| Metric | Custom CNN | ResNet-50 |
|---|---|---|
| **Top-1 Accuracy** | **~97.8 %** | **~99.2 %** |
| **Top-5 Accuracy** | ~99.7 % | ~99.96 % |
| **Parameters** | ~0.46 M | ~23.6 M |

**ResNet-50 beats the human benchmark (98.84 %).**

The small Custom CNN is still impressive: it reaches ~97.8 % with **50× fewer parameters**, which makes it a great lightweight option for devices with limited resources.

</div>


<div style="background:white;padding:22px;font-size:18px;line-height:1.65;border-left:5px solid #0b3d91;">

## **11. Conclusion**

- We trained **two deep-learning models** on the GTSRB traffic-sign dataset.
- Our **Custom CNN** — designed from scratch — already reaches **~97.8 %** accuracy.
- **ResNet-50 with transfer learning** pushes accuracy to **~99.2 %**, **above human level**.
- **Key takeaway:** *Transfer learning gives higher accuracy with less training, but a well-designed small CNN is still a strong, lightweight alternative.*

### **Thank you! &nbsp; Questions?**

</div>
